In [1]:
# @title Github Data Loader, Tokenizer Training, and Dataset Builder

import os

try:
    import google.colab
    REPO_URL = "https://github.com/wtheisen/nd-cse-10124-lectures.git"

    REPO_NAME = "/content/nd-cse-10124-lectures"
    L_PATH = "nd-cse-10124-lectures"

    %cd /content/
    !rm -r {REPO_NAME}

    # Clone repo
    if not os.path.exists(REPO_NAME):
        !git clone {REPO_URL}

        # cd into the data folder
        %cd {L_PATH}
        !pwd

except ImportError:
    print("Unable to download repo, either:")
    print("\tA.) You're not on colab")
    print("\tB.) It has already been cloned")

!pwd

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

import irishGPT as iGPT
r_t = iGPT.tokenizer.Regex_Tokenizer()
r_t.load('Datasets/openweb10k_vocab.json')

dataset = iGPT.dataset.IrishChatDataset('Datasets/openweb10k.txt', r_t)

/content
rm: cannot remove '/content/nd-cse-10124-lectures': No such file or directory
Cloning into 'nd-cse-10124-lectures'...
remote: Enumerating objects: 396, done.
remote: Counting objects: 100% (129/129), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 396 (delta 75), reused 96 (delta 45), pack-reused 267 (from 1)
Receiving objects: 100% (396/396), 34.32 MiB | 27.98 MiB/s, done.
Resolving deltas: 100% (251/251), done.
/content/nd-cse-10124-lectures
/content/nd-cse-10124-lectures
/content/nd-cse-10124-lectures
device: cuda


In [3]:
import torch
from irishGPT.irishChat import IrishChat
import irishGPT.utilities as uts
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch.nn.functional as F
from torch.utils.data import Dataset
import irishGPT.tokenizer as tokenizer

class IrishChatDataset(Dataset):
    def __init__(self, training_file):
        self.tokenizer = tokenizer.Regex_Tokenizer()
        self.tokenizer.train(uts.get_file_as_string(training_file), 512)
        self.device = torch.device('cuda' if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

        self.data = []
        for x in uts.get_file_as_list_strs(training_file, special_tokens=True):
            tokens = torch.tensor(self.tokenizer.encode(x), dtype=torch.long)
            if len(tokens) >= 2:
                # X = tokens[:-1], Y = tokens[1:]  (next-token prediction)
                self.data.append((tokens[:-1], tokens[1:]))
        self.padding_idx = 256

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

    # ---------- Collate (pad, optional one-hot, move to device) ----------
    def collate(self, batch):
        X_list, Y_list = zip(*batch)  # tuples of 1D tensors
        X = pad_sequence(X_list, batch_first=True, padding_value=self.padding_idx)        # (B,T)
        Y_idx = pad_sequence(Y_list, batch_first=True, padding_value=self.padding_idx)    # (B,T)

        Y = F.one_hot(Y_idx.clamp_min(0), num_classes=len(self.tokenizer.vocab)).float()        # (B,T,V)
        return X.to(self.device), Y.to(self.device)

if __name__ == "__main__":
    chat = IrishChat()
    dataset = IrishChatDataset("Datasets/zoomer.txt")
    train_loader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=dataset.collate)
    chat.train(train_loader, 100, 0.01, verbose=True)


Epoch 1/100
loss: 18.53244
accuracy: 0.02849
Output test:
------------------------------
Epoch 11/100
loss: 4.27274
accuracy: 0.18096
Output test:
------------------------------
Epoch 21/100
loss: 3.65300
accuracy: 0.24333
Output test:
------------------------------
Epoch 31/100
loss: 3.25605
accuracy: 0.28988
Output test:
------------------------------
Epoch 41/100
loss: 2.90641
accuracy: 0.34310
Output test:
------------------------------
Epoch 51/100
loss: 2.62790
accuracy: 0.38588
Output test:
------------------------------
Epoch 61/100
loss: 2.33672
accuracy: 0.44224
Output test:
------------------------------
Epoch 71/100
loss: 2.07534
accuracy: 0.49560
Output test:
------------------------------
Epoch 81/100
loss: 1.81652
accuracy: 0.55373
Output test:
------------------------------
Epoch 91/100
loss: 1.59906
accuracy: 0.60443
Output test:
------------------------------


In [4]:
# Mini ChatGPT-style UI for Colab (ipywidgets)
# Assumes you already have:
#   - dataset.tokenizer with encode/decode
#   - chat.chat(prompt_tokens, max_new_tokens=..., temperature=...)

!pip -q install ipywidgets

import ipywidgets as widgets
from IPython.display import display, HTML
import html

# --------- STATE ----------
history = []  # list of (role, text), role in {"user","assistant"}

# --------- STYLES ----------
style = HTML("""
<style>
.chat-wrap { font-family: system-ui, -apple-system, Segoe UI, Roboto, sans-serif; }
.chat-log  { height: 420px; overflow-y: auto; border: 1px solid #ddd; border-radius: 14px; padding: 12px; background: #fafafa; }
.msg { display: flex; margin: 10px 0; }
.bubble { max-width: 85%; padding: 10px 12px; border-radius: 14px; line-height: 1.35; white-space: pre-wrap; }
.user { justify-content: flex-end; }
.user .bubble { background: #dbeafe; border: 1px solid #bfdbfe; }
.assistant { justify-content: flex-start; }
.assistant .bubble { background: #ffffff; border: 1px solid #e5e7eb; }
.meta { font-size: 12px; color: #6b7280; margin: 0 0 6px 0; }
.row { display: flex; gap: 8px; margin-top: 10px; align-items: center; }
</style>
""")

# --------- UI ELEMENTS ----------
log = widgets.HTML(value="")
log_box = widgets.VBox([log], layout=widgets.Layout(width="100%"))
log_container = widgets.HTML(value="")  # unused; kept for clarity

prompt = widgets.Text(
    placeholder="Message IrishGPT…",
    layout=widgets.Layout(width="100%")
)
send = widgets.Button(description="Send", button_style="primary")
clear_btn = widgets.Button(description="Clear", button_style="")
temp = widgets.FloatSlider(value=0.8, min=0.1, max=1.5, step=0.05, description="Temp", continuous_update=False)
max_new = widgets.IntSlider(value=200, min=16, max=512, step=16, description="MaxNew", continuous_update=False)

status = widgets.HTML(value="<span class='meta'>Ready.</span>")

# --------- RENDERING ----------
def render_history():
    parts = ["<div class='chat-wrap'><div class='chat-log' id='chatlog'>"]
    for role, text in history:
        safe = html.escape(text)
        cls = "user" if role == "user" else "assistant"
        parts.append(f"<div class='msg {cls}'><div class='bubble'>{safe}</div></div>")
    parts.append("</div></div>")
    log.value = "".join(parts)

def add_message(role, text):
    history.append((role, text))
    render_history()

# --------- GENERATION ----------
def generate_reply(user_text: str) -> str:
    prompt_tokens = dataset.tokenizer.encode("<|sos|>" + user_text + "<|eos|>")[:-1]
    out_tokens = chat.chat(
        prompt_tokens,
        max_new_tokens=int(max_new.value),
        temperature=float(temp.value),
    )
    return dataset.tokenizer.decode(out_tokens)

# --------- HANDLERS ----------
def on_send(_=None):
    user_text = prompt.value.strip()
    if not user_text:
        return
    prompt.value = ""

    add_message("user", user_text)
    status.value = "<span class='meta'>Generating…</span>"

    try:
        reply = generate_reply(user_text)
        add_message("assistant", reply)
        status.value = "<span class='meta'>Ready.</span>"
    except Exception as e:
        add_message("assistant", f"[error] {type(e).__name__}: {e}")
        status.value = "<span class='meta'>Error.</span>"

def on_clear(_=None):
    history.clear()
    render_history()
    status.value = "<span class='meta'>Cleared.</span>"

send.on_click(on_send)
clear_btn.on_click(on_clear)
prompt.on_submit(on_send)

# --------- LAYOUT ----------
controls = widgets.HBox([send, clear_btn, temp, max_new], layout=widgets.Layout(width="100%"))
ui = widgets.VBox([style, log_box, widgets.HBox([prompt]), controls, status], layout=widgets.Layout(width="100%"))

render_history()
display(ui)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 51.1 MB/s eta 0:00:00


TraitError: The 'children' trait of a VBox instance contains an Instance of a TypedTuple which expected a Widget, not the HTML <IPython.core.display.HTML object>.

In [7]:
# Mini ChatGPT-style UI for Colab (ipywidgets) — FIXED
# Assumes you already have:
#   - dataset.tokenizer with encode/decode
#   - chat.chat(prompt_tokens, max_new_tokens=..., temperature=...)

!pip -q install ipywidgets

from google.colab import output
output.enable_custom_widget_manager()

import ipywidgets as widgets
import html

# --------- STATE ----------
history = []  # list of (role, text), role in {"user","assistant"}

# --------- WIDGET STYLES (must be widgets.HTML, not IPython.display.HTML) ----------
style = widgets.HTML(value="""
<style>
.chat-wrap { font-family: system-ui, -apple-system, Segoe UI, Roboto, sans-serif; }
.chat-log  { height: 420px; overflow-y: auto; border: 1px solid #ddd; border-radius: 14px; padding: 12px; background: #fafafa; }
.msg { display: flex; margin: 10px 0; }
.bubble { max-width: 85%; padding: 10px 12px; border-radius: 14px; line-height: 1.35; white-space: pre-wrap; }
.user { justify-content: flex-end; }
.user .bubble { background: #dbeafe; border: 1px solid #bfdbfe; }
.assistant { justify-content: flex-start; }
.assistant .bubble { background: #ffffff; border: 1px solid #e5e7eb; }
.meta { font-size: 12px; color: #6b7280; margin-top: 6px; }
.row { display: flex; gap: 8px; margin-top: 10px; align-items: center; }
</style>
""")

# --------- UI ELEMENTS ----------
log = widgets.HTML(value="")
prompt = widgets.Text(
    placeholder="Message IrishGPT…",
    layout=widgets.Layout(width="60%")
)
send = widgets.Button(description="Send", button_style="primary")
clear_btn = widgets.Button(description="Clear")

temp = widgets.FloatSlider(
    value=0.8, min=0.1, max=1.5, step=0.05,
    description="Temp", continuous_update=False,
    layout=widgets.Layout(width="300px")
)

max_new = widgets.IntSlider(
    value=200, min=16, max=512, step=16,
    description="MaxNew", continuous_update=False,
    layout=widgets.Layout(width="320px")
)

status = widgets.HTML(value="<div class='meta'>Ready.</div>")

# --------- RENDERING ----------
def render_history():
    parts = ["<div class='chat-wrap'><div class='chat-log'>"]
    for role, text in history:
        safe = html.escape(text)
        cls = "user" if role == "user" else "assistant"
        parts.append(f"<div class='msg {cls}'><div class='bubble'>{safe}</div></div>")
    parts.append("</div></div>")
    log.value = "".join(parts)

def add_message(role, text):
    history.append((role, text))
    render_history()

# --------- GENERATION ----------
def generate_reply(user_text: str) -> str:
    prompt_tokens = dataset.tokenizer.encode("<|sos|>" + user_text + "<|eos|>")[:-1]
    out_tokens = chat.chat(
        prompt_tokens,
        max_new_tokens=int(max_new.value),
        temperature=float(temp.value),
    )
    return dataset.tokenizer.decode(out_tokens)

# --------- HANDLERS ----------
def on_send(_=None):
    user_text = prompt.value.strip()
    if not user_text:
        return
    prompt.value = ""

    add_message("user", user_text)
    status.value = "<div class='meta'>Generating…</div>"

    try:
        reply = generate_reply(user_text)
        add_message("assistant", reply)
        status.value = "<div class='meta'>Ready.</div>"
    except Exception as e:
        add_message("assistant", f"[error] {type(e).__name__}: {e}")
        status.value = "<div class='meta'>Error.</div>"

def on_clear(_=None):
    history.clear()
    render_history()
    status.value = "<div class='meta'>Cleared.</div>"

send.on_click(on_send)
clear_btn.on_click(on_clear)
prompt.on_submit(on_send)

# --------- LAYOUT ----------
controls = widgets.HBox(
    [send, clear_btn, temp, max_new],
    layout=widgets.Layout(width="100%")
)

ui = widgets.VBox(
    [style, log, prompt, controls, status],
    layout=widgets.Layout(
        width="40%",      # ← chat width
        margin="0 auto"   # ← center horizontally
    )
)


render_history()
display(ui)